# Session 2: Network Cables, Connectors and Wiring Standards

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamnakhan123/Cybersecurity-and-Digital-Forensics/blob/main/term-1/03-computer-networks/code/session-02-cables-connectors-wiring.ipynb)

*Cybersecurity and Digital Forensics* · Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom (Shift+Enter). Every example is self-contained and safe to run in a lab or on Colab.

## `session-02_wiremap.py`

In [1]:
"""Session 2 - T568A / T568B pin maps and a simulated cable wire-map tester.

A real cable tester sends a signal down each of the 8 wires and reports where it
arrives. Here we describe a cable as "end A standard" + "end B standard" (or a
custom pin map with faults) and classify it like a tester would.
Run:  python session-02_wiremap.py
"""

T568A = ["white/green", "green", "white/orange", "blue",
         "white/blue", "orange", "white/brown", "brown"]
T568B = ["white/orange", "orange", "white/green", "blue",
         "white/blue", "green", "white/brown", "brown"]


def pin_map(end_a: list[str], end_b: list[str]) -> list[int | None]:
    """For each pin 1-8 on end A, which pin on end B carries the same wire colour?"""
    result = []
    for colour in end_a:
        result.append(end_b.index(colour) + 1 if colour in end_b else None)
    return result


def classify(mapping: list[int | None]) -> str:
    straight = list(range(1, 9))
    crossover = [3, 6, 1, 4, 5, 2, 7, 8]       # 1<->3 and 2<->6 swapped
    rollover = list(range(8, 0, -1))           # console cable: 1<->8, 2<->7 ...
    if None in mapping:
        return "OPEN: pin(s) " + ", ".join(str(i + 1) for i, m in enumerate(mapping) if m is None)
    if mapping == straight:
        return "straight-through (PC to switch)"
    if mapping == crossover:
        return "crossover (switch to switch, PC to PC on old gear)"
    if mapping == rollover:
        return "rollover (console cable)"
    # pairs 1-2, 3-6, 4-5, 7-8 must stay together or the twist is lost
    pairs = [(1, 2), (3, 6), (4, 5), (7, 8)]
    for a, b in pairs:
        if {mapping[a - 1], mapping[b - 1]} not in [{a, b}, {1, 2}, {3, 6}, {4, 5}, {7, 8}]:
            return f"SPLIT PAIR or mis-wire around pins {a}/{b}"
    return "REVERSED/CROSSED pair: " + str(mapping)


if __name__ == "__main__":
    print("Pin  T568A           T568B")
    for i, (a, b) in enumerate(zip(T568A, T568B), 1):
        mark = "  <- differs" if a != b else ""
        print(f" {i}   {a:<15} {b:<15}{mark}")

    print("\nTesting cables:")
    tests = {
        "B to B": (T568B, T568B),
        "A to A": (T568A, T568A),
        "A to B": (T568A, T568B),
        "B to reversed B": (T568B, list(reversed(T568B))),
    }
    broken = T568B.copy()
    broken[5] = "cut"                         # green conductor broken at pin 6
    tests["B with cut green wire"] = (T568B, broken)
    swapped = T568B.copy()
    swapped[2], swapped[3] = swapped[3], swapped[2]   # installer swapped pins 3 and 4
    tests["B with pins 3/4 swapped"] = (T568B, swapped)

    for name, (a, b) in tests.items():
        m = pin_map(a, b)
        print(f"  {name:<24} map={m}  -> {classify(m)}")

Pin  T568A           T568B
 1   white/green     white/orange     <- differs
 2   green           orange           <- differs
 3   white/orange    white/green      <- differs
 4   blue            blue           
 5   white/blue      white/blue     
 6   orange          green            <- differs
 7   white/brown     white/brown    
 8   brown           brown          

Testing cables:
  B to B                   map=[1, 2, 3, 4, 5, 6, 7, 8]  -> straight-through (PC to switch)
  A to A                   map=[1, 2, 3, 4, 5, 6, 7, 8]  -> straight-through (PC to switch)
  A to B                   map=[3, 6, 1, 4, 5, 2, 7, 8]  -> crossover (switch to switch, PC to PC on old gear)
  B to reversed B          map=[8, 7, 6, 5, 4, 3, 2, 1]  -> rollover (console cable)
  B with cut green wire    map=[1, 2, 3, 4, 5, None, 7, 8]  -> OPEN: pin(s) 6
  B with pins 3/4 swapped  map=[1, 2, 4, 3, 5, 6, 7, 8]  -> SPLIT PAIR or mis-wire around pins 3/6


## `session-02_cable_selector.py`

In [2]:
"""Session 2 - Choose a cable type for a link from distance, speed and environment.

The table holds typical *standard* limits (not vendor marketing maximums).
Run:  python session-02_cable_selector.py
"""

# name, max metres, max Gbps, immune to EMI?, relative cost (1 = cheapest)
CABLES = [
    ("Cat5e UTP",              100,   1, False, 1),
    ("Cat6 UTP",               100,  10, False, 2),   # 10G only up to ~55 m
    ("Cat6a STP",              100,  10, False, 3),
    ("Cat8 STP",                30,  40, False, 5),
    ("Multimode fibre OM3",    300,  10, True,  4),
    ("Multimode fibre OM4",    400,  10, True,  5),
    ("Single-mode fibre OS2", 10000, 100, True,  6),
]


def choose(distance_m: float, speed_gbps: float, high_emi: bool) -> list[str]:
    """Return suitable cables, cheapest first, with a reason for each rejection."""
    ok = []
    for name, max_m, max_g, emi_immune, cost in CABLES:
        limit_m = max_m
        if name == "Cat6 UTP" and speed_gbps > 1:
            limit_m = 55                                  # Cat6 at 10G
        if distance_m > limit_m or speed_gbps > max_g:
            continue
        if high_emi and not emi_immune and "STP" not in name:
            continue                                      # unshielded copper near motors = errors
        ok.append((cost, name))
    return [n for _, n in sorted(ok)]


if __name__ == "__main__":
    jobs = [
        ("Office desk to wall jack", 40, 1, False),
        ("Server rack to rack, 10G", 25, 10, False),
        ("Floor 1 to floor 5 riser, 10G", 180, 10, False),
        ("Factory floor next to motors", 70, 1, True),
        ("Campus building A to B", 2200, 10, False),
        ("Desk to switch 140 m away", 140, 1, False),
    ]
    for desc, d, s, emi in jobs:
        picks = choose(d, s, emi)
        best = picks[0] if picks else "NO single cable: add a switch or use fibre"
        print(f"{desc:<32} {d:>5} m {s:>3} Gbps EMI={'yes' if emi else 'no ':<3} -> {best}")

Office desk to wall jack            40 m   1 Gbps EMI=no  -> Cat5e UTP
Server rack to rack, 10G            25 m  10 Gbps EMI=no  -> Cat6 UTP
Floor 1 to floor 5 riser, 10G      180 m  10 Gbps EMI=no  -> Multimode fibre OM3
Factory floor next to motors        70 m   1 Gbps EMI=yes -> Cat6a STP
Campus building A to B            2200 m  10 Gbps EMI=no  -> Single-mode fibre OS2
Desk to switch 140 m away          140 m   1 Gbps EMI=no  -> Multimode fibre OM3
